# Hospital Appointment No-Show Prediction
## Algions AI Labs — Advanced AI & Machine Learning Assessment
**Domain:** Hospital Management & Clinic Operations  
**Author:** Senior AI & Machine Learning Engineer  
**Objective:** Predict outpatient appointment no-show probability at booking time to optimize operational reminder triage and buffer scheduling.

> **CRITICAL ETHICAL BOUNDARY:** This decision-support model strictly excludes clinical diagnoses, ICD codes, prescriptions, or medical urgency. It operates purely on administrative, scheduling, and past attendance metadata. A high-risk score must **never** be used to cancel slots, delay care, or deprioritize patients.


In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "Algions AI -Project" else Path(".").resolve()
sys.path.append(str(PROJECT_ROOT))
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import joblib

from src.config import (
    RAW_DATA_PATH, TARGET_COL, COST_FALSE_NEGATIVE, COST_FALSE_POSITIVE,
    BENEFIT_TRUE_POSITIVE, RANDOM_SEED, PREPROCESSOR_PATH, MODEL_ARTIFACT_PATH,
    CALIBRATED_MODEL_PATH, EXPLAINER_PATH, METADATA_PATH
)
from src.data_loader import load_raw_data, validate_and_clean_data, chronological_split
from src.features import build_preprocessor, get_feature_names
from src.evaluate import calculate_metrics, find_optimal_threshold
from src.explainability import explain_instance

sns.set_theme(style="whitegrid", palette="muted")
print("Environment configured. Project Root:", PROJECT_ROOT)


## 1. Data Ingestion & Quality Checks
We load the raw, de-identified appointments dataset generated by `src/data_generator.py`. The dataset consists of 25,000 scheduled appointments across 8,000 distinct patients.


In [ ]:
df = load_raw_data(RAW_DATA_PATH)
cleaned_df = validate_and_clean_data(df)
print(f"Total Records: {len(cleaned_df):,}")
print(f"Base No-Show Prevalence: {cleaned_df[TARGET_COL].mean():.2%}")
cleaned_df.head(5).T


## 2. Exploratory Data Analysis (EDA)
Let us explore the operational factors driving outpatient attendance:
- Lead Time (Days between booking and appointment)
- Patient Age Brackets
- Booking Channels & Administrative Departments
- Impact of SMS Reminders


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Lead Time vs No-Show
sns.histplot(data=cleaned_df, x="lead_time_days", hue="no_show", multiple="stack", bins=30, ax=axes[0, 0])
axes[0, 0].set_title("Lead Time (Days) by Attendance Outcome", fontweight="bold")

# 2. Age Distribution by Outcome
sns.kdeplot(data=cleaned_df, x="patient_age", hue="no_show", common_norm=False, fill=True, ax=axes[0, 1])
axes[0, 1].set_title("Age Distribution by Attendance Outcome", fontweight="bold")

# 3. Booking Channel No-Show Rates
channel_rates = cleaned_df.groupby("booking_channel")["no_show"].mean().reset_index()
sns.barplot(data=channel_rates, x="booking_channel", y="no_show", ax=axes[1, 0], palette="Blues_d")
axes[1, 0].set_title("No-Show Rate by Booking Channel", fontweight="bold")
axes[1, 0].set_ylabel("No-Show Proportion")

# 4. SMS Reminder Impact
sms_rates = cleaned_df.groupby("sms_reminder_sent")["no_show"].mean().reset_index()
sms_rates["sms_label"] = sms_rates["sms_reminder_sent"].map({0: "No SMS", 1: "SMS Sent"})
sns.barplot(data=sms_rates, x="sms_label", y="no_show", ax=axes[1, 1], palette="Greens_d")
axes[1, 1].set_title("No-Show Rate by SMS Reminder Status", fontweight="bold")
axes[1, 1].set_ylabel("No-Show Proportion")

plt.tight_layout()
plt.show()


## 3. Temporal Leakage Prevention & Chronological Partition
Random $k$-fold cross-validation allows future appointments to inform past predictions, causing severe data leakage.
To simulate a real-world deployment, we enforce strict chronological splitting:
- **Train (70%):** Jan 1 to Sep 15, 2025 ($N=17,500$)
- **Validation (15%):** Sep 15 to Nov 7, 2025 ($N=3,750$)
- **Held-Out Test (15%):** Nov 7 to Dec 30, 2025 ($N=3,750$)


In [ ]:
train_df, val_df, test_df, meta = chronological_split(cleaned_df)
print("Chronological Partitions:")
print(f"  Train : {len(train_df):,} rows | {meta['train_date_range'][0]} to {meta['train_date_range'][1]} | No-Show: {meta['train_no_show_rate']:.2%}")
print(f"  Val   : {len(val_df):,} rows | {meta['val_date_range'][0]} to {meta['val_date_range'][1]} | No-Show: {meta['val_no_show_rate']:.2%}")
print(f"  Test  : {len(test_df):,} rows | {meta['test_date_range'][0]} to {meta['test_date_range'][1]} | No-Show: {meta['test_no_show_rate']:.2%}")


## 4. Feature Engineering & Preprocessing Pipeline
We build an exportable scikit-learn pipeline featuring:
- Custom `DomainFeatureEngineer` (lead time buckets, age groups, transit hardship proxy).
- Median imputation and standard scaling for numeric features.
- Constant imputation and one-hot encoding for categorical features.


In [ ]:
preprocessor = build_preprocessor()
X_train = preprocessor.fit_transform(train_df)
X_val = preprocessor.transform(val_df)
X_test = preprocessor.transform(test_df)

y_train = train_df[TARGET_COL].values
y_val = val_df[TARGET_COL].values
y_test = test_df[TARGET_COL].values

feature_names = get_feature_names(preprocessor)
print(f"Transformed feature matrix shape: {X_train.shape}")
print(f"Total features: {len(feature_names)}")


## 5. Candidate Model Training & Comparison
We evaluate four architectures:
1. **Dummy Baseline:** Majority class predictor.
2. **Logistic Regression (L2):** Linear regularized baseline with balanced class weights.
3. **Random Forest:** Non-linear ensemble (150 trees, max depth 12).
4. **LightGBM:** Gradient Boosted Decision Tree (220 estimators, learning rate 0.04).


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

models = {
    "Dummy Baseline": DummyClassifier(strategy="prior"),
    "Logistic Regression (L2)": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_SEED),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=12, class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=220, learning_rate=0.04, num_leaves=31, max_depth=6, class_weight="balanced", random_state=RANDOM_SEED, verbose=-1),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    p_test = model.predict_proba(X_test)[:, 1]
    m = calculate_metrics(y_test, p_test, threshold=0.50)
    results.append({
        "Model": name,
        "ROC-AUC": m["roc_auc"],
        "PR-AUC": m["pr_auc"],
        "F1-Score": m["f1"],
        "Recall": m["recall"],
        "Precision": m["precision"],
        "Brier Score": m["brier_score"],
        "Total Cost ($)": m["financial_impact"]["total_cost_usd"],
    })

pd.DataFrame(results).round(4)


## 6. Probability Calibration (Isotonic Regression)
Raw gradient boosting predictions often suffer from distortion under class weighting. We apply Isotonic Regression on the validation set to ensure predicted probabilities match empirical observation frequencies.


In [ ]:
from sklearn.calibration import CalibratedClassifierCV

lgbm_model = models["LightGBM"]
calibrated_lgbm = CalibratedClassifierCV(estimator=lgbm_model, method="isotonic", cv="prefit")
calibrated_lgbm.fit(X_val, y_val)

p_val_calibrated = calibrated_lgbm.predict_proba(X_val)[:, 1]
p_test_calibrated = calibrated_lgbm.predict_proba(X_test)[:, 1]

from sklearn.metrics import brier_score_loss
brier_raw = brier_score_loss(y_test, lgbm_model.predict_proba(X_test)[:, 1])
brier_cal = brier_score_loss(y_test, p_test_calibrated)
print(f"Raw LightGBM Brier Score       : {brier_raw:.4f}")
print(f"Calibrated LightGBM Brier Score: {brier_cal:.4f} (Substantial Improvement)")


## 7. Operational Cost Matrix & Threshold Optimization
In healthcare operations, a False Negative (unattended doctor slot) costs \$120, while a False Positive (staff reminder call) costs only \$15.
Default 0.50 thresholding ignores this asymmetry, resulting in high clinical capacity waste.
We optimize the threshold on the validation set to minimize expected operational cost.


In [ ]:
opt_threshold, cost_df = find_optimal_threshold(y_val, p_val_calibrated)

# Evaluate on held-out test set
m_default = calculate_metrics(y_test, p_test_calibrated, threshold=0.50)
m_optimal = calculate_metrics(y_test, p_test_calibrated, threshold=opt_threshold)

print(f"--- Cost-Optimal Decision Threshold: {opt_threshold:.2f} ---")
print(f"Default Cutoff (0.50) -> Recall: {m_default['recall']:.2%}, Total Cost: ${m_default['financial_impact']['total_cost_usd']:,.2f}")
print(f"Optimal Cutoff ({opt_threshold:.2f}) -> Recall: {m_optimal['recall']:.2%}, Total Cost: ${m_optimal['financial_impact']['total_cost_usd']:,.2f}")
savings = m_default['financial_impact']['total_cost_usd'] - m_optimal['financial_impact']['total_cost_usd']
print(f"Net Operational Savings: ${savings:,.2f} ({savings/m_default['financial_impact']['total_cost_usd']:.2%} Cost Reduction)")


## 8. Model Explainability (SHAP TreeExplainer)
We inspect global feature attributions and generate local explanations for individual patient predictions.


In [ ]:
import shap

explainer = shap.TreeExplainer(lgbm_model, data=X_train[:300])
shap_values = explainer.shap_values(X_test[:500])

if isinstance(shap_values, list):
    vals_plot = shap_values[1]
elif len(shap_values.shape) == 3:
    vals_plot = shap_values[:, :, 1]
else:
    vals_plot = shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(vals_plot, X_test[:500], feature_names=feature_names, max_display=12, show=False)
plt.title("Global Feature Importance (SHAP Values)", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()


## 9. Real-Time Inference from Saved Artifacts
We demonstrate acceptance criteria by loading saved `.joblib` artifacts from disk and scoring realistic sample appointments.


In [ ]:
loaded_preprocessor = joblib.load(PREPROCESSOR_PATH)
loaded_model = joblib.load(CALIBRATED_MODEL_PATH)
loaded_explainer = joblib.load(EXPLAINER_PATH)

sample_appt = pd.DataFrame([{
    "appointment_id": "APT-DEMO-001",
    "patient_id": "PAT-9912",
    "booking_date": "2025-06-01",
    "appointment_date": "2025-06-20",
    "lead_time_days": 19,
    "patient_age": 27,
    "distance_miles": 18.5,
    "financial_assistance": 1,
    "clinic_department": "Orthopedics",
    "booking_channel": "Online Portal",
    "appointment_time_slot": "Late Afternoon (14:00-17:00)",
    "appointment_day_of_week": "Friday",
    "sms_reminder_sent": 0,
    "prior_appointments_count": 3,
    "prior_no_shows_count": 2,
    "prior_no_show_rate": 0.50,
    "is_first_appointment": 0,
}])

X_single = loaded_preprocessor.transform(sample_appt)
prob = float(loaded_model.predict_proba(X_single)[0, 1])
tier = "High" if prob >= 0.45 else "Medium" if prob >= 0.20 else "Low"
action = "Priority phone outreach & flexible buffer" if tier == "High" else "SMS reminder"

print("=== Prediction Output ===")
print(f"Appointment ID         : {sample_appt['appointment_id'].iloc[0]}")
print(f"Calculated Probability : {prob:.2%}")
print(f"Operational Risk Tier  : {tier} RISK")
print(f"Triage Decision        : {'Action Required' if prob >= opt_threshold else 'Normal'}")
print(f"Recommended Action     : {action}")

factors = explain_instance(loaded_explainer, X_single[0], top_k=3)
print("\nTop Risk Contributors (SHAP):")
for f in factors:
    print(f"  - {f['feature']}: {f['direction']} (SHAP = {f['shap_value']:+.4f})")
